# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

**Dataset Source**: FAIR² mental health survey, described in Croissant schema format.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\nDescription: {metadata.description}")
print(f"License: {metadata.license}\nVersion: {metadata.version}\nDate Published: {metadata.datePublished}")
print(f"Identifiers: {metadata.identifier}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

To fully explore the structure, we'll list all record sets, their `@id`, fields, and columns, referencing strictly by `@id` as required.

In [ ]:
# Examine the record sets

# The recordSet attribute is a list of mlcroissant.RecordSet objects
record_sets = metadata.recordSet

# List available record sets and their fields/columns
for rs in record_sets:
    print(f"Record Set Name: {getattr(rs, 'name', 'N/A')} | @id: {rs['@id'] if isinstance(rs, dict) else rs._id}")
    # List fields
    if hasattr(rs, 'field') and rs.field:
        print("  Fields:")
        for fld in rs.field:
            field_id = fld['@id'] if isinstance(fld, dict) else fld._id
            field_name = getattr(fld, 'name', 'N/A')
            print(f"    - {field_name} (@id: {field_id})")
    # List columns
    if hasattr(rs, 'column') and rs.column:
        print("  Columns:")
        for col in rs.column:
            col_id = col['@id'] if isinstance(col, dict) else col._id
            col_name = getattr(col, 'name', 'N/A')
            print(f"    - {col_name} (@id: {col_id})")
    print("----")

# For demonstration, print a sample record from each record set
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else rs._id
    print(f"Sample from Record Set '@id': {rs_id}")
    try:
        # Print first record
        for rec in dataset.records(record_set=rs_id):
            pprint.pprint(rec)
            break
    except Exception as e:
        print(f"  Could not retrieve records: {e}")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis.
Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all record sets into DataFrames, referencing each by @id
dataframes = {}

# Build a list of record set @id's
record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else rs._id
    record_set_ids.append(rs_id)

for rs_id in record_set_ids:
    # Records generator
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set @id: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Failed to load records for @id {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field and demonstrate outlier filtering, normalization, and grouping. All fields are referenced by their `@id`.

In [ ]:
# Pick the main record set (usually the survey responses)
main_rs_id = record_set_ids[0] if record_set_ids else None
df_main = dataframes[main_rs_id]

# Review column @id's and names
print(f"Available fields for main record set (@id: {main_rs_id}):")
for col in df_main.columns:
    print(f"  - {col}")

# Let's suppose the dataset contains a numeric score field, e.g. GAD-7 total score
# This column's @id could be 'gad7_total_score'
# For demonstration, check if a likely numeric field exists, else pick one
numeric_field_id = None
possible_numeric_fields = ['gad7_total_score', 'phq9_total_score', 'psq_total_score', 'age']
for c in df_main.columns:
    if c in possible_numeric_fields:
        numeric_field_id = c
        break
if not numeric_field_id:
    numeric_field_id = df_main.select_dtypes(include='number').columns[0] if len(df_main.select_dtypes(include='number').columns) > 0 else df_main.columns[0]

print(f"Using numeric field '@id': {numeric_field_id}")

# Filter records with score above a threshold
threshold = 10
filtered_df = df_main[df_main[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field, e.g. 'level_of_education' (@id)
group_field_id = 'level_of_education' if 'level_of_education' in df_main.columns else df_main.columns[0]
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the numeric field and a grouped bar plot.

In [ ]:
# Simple visualizations using matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df_main[numeric_field_id], bins=15, kde=True)
plt.title(f"Distribution of '{numeric_field_id}' in main record set")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Bar plot of average score by education level
if 'grouped_df' in locals():
    plt.figure(figsize=(10,6))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.xticks(rotation=45)
    plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
We have loaded the FAIR² Mental Health Survey dataset with `mlcroissant`, explored its structure down to fields and columns (by `@id`), extracted the main survey records, and performed basic filtering, normalization, grouping, and visualizations. This data supports exploration of mental health trends and demographic factors in Kilifi County, Kenya. Further analysis can build on the structured and AI-ready format, referencing all elements by their `@id` for reproducibility and data provenance.